In [3]:
# Importing Libraries
import ast
import pandas as pd
from datasets import load_dataset
import matplotlib.pyplot as plt
plt.close('all')

import seaborn as sns

# Loading Data
dataset = load_dataset('lukebarousse/data_jobs')
df = dataset['train'].to_pandas()

# Data Cleanup
# make datetime string into datetime object
df['job_posted_date'] = pd.to_datetime(df['job_posted_date'])
# make string list in column job_skills into panda series (list) object
df["job_skills"] = df["job_skills"].apply(lambda skill_list: ast.literal_eval(skill_list) if pd.notna(skill_list) else None)

* show which companies are most hiring in the UK
* percentage of jobs are wfh
* breakdown of the top 5 jobs
* salary year average of data jobs
* monthly demand for jobs

* show which companies are most hiring in the UK

In [7]:
df_UK = df[df["job_country"] == "United Kingdom"].copy()

In [14]:
df_companies = df_UK.groupby("company_name").size()

df_companies = df_companies.reset_index(name="count")

df_companies.sort_values(by="count", ascending=False, inplace=True)

df_companies

,company_name,count
3727,Harnham,1222
4480,Jobleads-UK,671
1856,ClickJobs.io,664
1598,CareerAddict,324
6954,RemoteWorker UK,270
...,...,...
9627,webmaster,1
9626,waters.com,1
32,4Front Recruitment Limited,1
30,3PL,1


* percentage of jobs are wfh

In [32]:
df_work_from_home = df_UK.value_counts("job_work_from_home").reset_index(name="count")

df_work_from_home["total"] = df_work_from_home["count"].loc[0] + df_work_from_home["count"].loc[1]

df_work_from_home["percentage"] = (df_work_from_home["count"] / df_work_from_home["total"]) * 100

df_work_from_home

,job_work_from_home,count,total,percentage
0,False,36611,40375,90.677399
1,True,3764,40375,9.322601


* breakdown of the top 5 jobs

In [38]:
df_job_counts = df_UK.value_counts("job_title_short").sort_values(ascending=False).head().reset_index(name="count")

df_job_counts

,job_title_short,count
0,Data Engineer,11807
1,Data Analyst,10482
2,Data Scientist,9148
3,Senior Data Engineer,3337
4,Senior Data Scientist,2367


In [70]:
len(df["salary_year_avg"].unique())

106

* salary year average of data jobs

In [ ]:
# Fill na values in column salary_year_avg with the median salary of each job title
# Return series of median values
job_medians = df_UK.groupby("job_title_short")["salary_year_avg"].median()

# Map each NA value in column salary_year_avg with median value based on column job_title_short
# .transform("median") method does not work so use map with array job_medians passed through it
df_UK["salary_year_avg"] = (
    df_UK["salary_year_avg"]
    .fillna(df_UK["job_title_short"].map(job_medians))
)

In [ ]:
df_median_salaries = df_UK.groupby("job_title_short")["salary_year_avg"].median().sort_values(ascending=False)

df_median_salaries = df_median_salaries.reset_index(name="median_salary")

df_median_salaries

,job_title_short,median_salary
0,Senior Data Scientist,157500.0
1,Machine Learning Engineer,149653.0
2,Senior Data Engineer,147500.0
3,Senior Data Analyst,111175.0
4,Data Engineer,110000.0
5,Cloud Engineer,105300.0
6,Data Scientist,105300.0
7,Software Engineer,89100.0
8,Data Analyst,87750.0
9,Business Analyst,56700.0


* monthly demand for jobs